# Feature Engineering Validation


## Purpose
Validate all engineered features: correlation with series_winner, sanity checks, and feature importance preview.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys; sys.path.insert(0, str(Path(".").parent / "src"))
from nba_predictor.config import cfg


## Load Series Dataset

In [ ]:
series_path = cfg.project_root / "data/processed/series_dataset.parquet"
if series_path.exists():
    df = pd.read_parquet(series_path)
    print(f"Series dataset: {len(df)} rows, {len(df.columns)} columns")
    print("Target distribution:", df["higher_seed_wins"].value_counts())
    print("Series length distribution:", df["series_length"].value_counts())
else:
    print("Run make process first")


## Feature Correlations with higher_seed_wins

In [ ]:
from scipy import stats

Path("../reports/figures").mkdir(parents=True, exist_ok=True)

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
exclude = {"higher_seed_wins", "series_length", "season", "seed_diff",
           "home_court_advantage", "conference_East", "conference_West",
           "era_showtime", "era_defensive", "era_transition", "era_analytics", "season_flag"}
feat_cols = [c for c in numeric_cols if c not in exclude]

corr_rows = []
for col in feat_cols:
    valid = df[[col, "higher_seed_wins"]].dropna()
    if len(valid) < 20:
        continue
    r, p = stats.pointbiserialr(valid["higher_seed_wins"], valid[col])
    corr_rows.append({"feature": col, "r": r, "p": p, "n": len(valid)})

corr_df = pd.DataFrame(corr_rows).sort_values("r", key=abs, ascending=False)
corr_df["sig"] = corr_df["p"] < 0.05

fig, ax = plt.subplots(figsize=(8, max(6, len(corr_df) * 0.28)))
colors = ["steelblue" if sig else "lightgray" for sig in corr_df["sig"]]
ax.barh(corr_df["feature"], corr_df["r"], color=colors)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Point-biserial correlation with higher_seed_wins")
ax.set_title("Feature Correlations with Series Outcome\n(blue = p < 0.05)")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("../reports/figures/04_feature_correlations.png", dpi=120, bbox_inches="tight")
plt.show()

print("\nTop 10 features by |r|:")
print(corr_df.head(10)[["feature", "r", "p", "sig"]].to_string(index=False))


## Sanity Checks

In [ ]:
print("=== Sanity Checks ===\n")

# 1. delta_VORP positive for winners
winners = df[df["higher_seed_wins"] == 1]["delta_VORP"].dropna()
losers  = df[df["higher_seed_wins"] == 0]["delta_VORP"].dropna()
print(f"1. delta_VORP mean — higher seed wins: {winners.mean():+.3f},  upset: {losers.mean():+.3f}")
assert winners.mean() > losers.mean(), "FAIL: winners should have higher delta_VORP"
print("   PASS")

# 2. Win rate by seed_diff bucket
print("\n2. Higher-seed win rate by seed_diff:")
for sd in sorted(df["seed_diff"].dropna().unique()):
    sub = df[df["seed_diff"] == sd]
    wr = sub["higher_seed_wins"].mean()
    print(f"   seed_diff={int(sd):2d}  win_rate={wr:.1%}  n={len(sub)}")

# 3. delta_NRtg positive on average for winners
dn_w = df[df["higher_seed_wins"] == 1]["delta_NRtg"].dropna()
dn_l = df[df["higher_seed_wins"] == 0]["delta_NRtg"].dropna()
print(f"\n3. delta_NRtg mean — winners: {dn_w.mean():+.3f},  upsets: {dn_l.mean():+.3f}")
assert dn_w.mean() > dn_l.mean(), "FAIL"
print("   PASS")

# 4. Roster_VORP_available_pct range
rvap = df["higher_Roster_VORP_available_pct"].dropna()
print(f"\n4. higher_Roster_VORP_available_pct range: [{rvap.min():.3f}, {rvap.max():.3f}]")
if rvap.min() >= 0 and rvap.max() <= 1.0:
    print("   PASS (in [0,1])")
else:
    print("   NOTE: values outside [0,1] — check injury pipeline")

# 5. higher_seed_wins overall
overall_wr = df["higher_seed_wins"].mean()
print(f"\n5. Overall higher-seed win rate: {overall_wr:.1%}  (naive accuracy benchmark)")


## Momentum Feature Validation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# L10_NRtg_delta: do teams trending up win more?
col = "delta_L10_NRtg"
valid = df[[col, "higher_seed_wins"]].dropna()
# Bucket into terciles
valid["bucket"] = pd.qcut(valid[col], q=3, labels=["Bottom 3rd", "Middle 3rd", "Top 3rd"])
win_by_bucket = valid.groupby("bucket")["higher_seed_wins"].agg(["mean", "count"])

ax = axes[0]
ax.bar(win_by_bucket.index, win_by_bucket["mean"], color=["tomato", "gold", "steelblue"])
ax.axhline(df["higher_seed_wins"].mean(), color="black", ls="--", lw=1.5,
           label=f"Overall: {df['higher_seed_wins'].mean():.1%}")
ax.set_ylim(0, 1)
ax.set_ylabel("Higher-seed win rate")
ax.set_xlabel("Momentum advantage tercile (delta_L10_NRtg)")
ax.set_title("Momentum Advantage vs Series Win Rate")
for i, (idx, row) in enumerate(win_by_bucket.iterrows()):
    ax.text(i, row["mean"] + 0.02, f"n={int(row['count'])}", ha="center", fontsize=9)
ax.legend()

# delta_NRtg vs delta_L10_NRtg scatter
ax2 = axes[1]
delta_nrtg = df["delta_NRtg"].dropna()
delta_l10  = df["delta_L10_NRtg"].dropna()
common_idx = delta_nrtg.index.intersection(delta_l10.index)
colors_pt = ["steelblue" if w == 1 else "tomato" for w in df.loc[common_idx, "higher_seed_wins"]]
ax2.scatter(df.loc[common_idx, "delta_NRtg"], df.loc[common_idx, "delta_L10_NRtg"],
            c=colors_pt, alpha=0.5, s=30)
ax2.axhline(0, color="gray", lw=0.8); ax2.axvline(0, color="gray", lw=0.8)
ax2.set_xlabel("Season delta_NRtg"); ax2.set_ylabel("delta_L10_NRtg (momentum)")
ax2.set_title("Season Strength vs Late-Season Momentum\n(blue=higher seed wins, red=upset)")
from matplotlib.patches import Patch
ax2.legend(handles=[Patch(color="steelblue", label="Higher seed wins"),
                    Patch(color="tomato", label="Upset")], fontsize=9)

plt.tight_layout()
plt.savefig("../reports/figures/04_momentum_validation.png", dpi=120, bbox_inches="tight")
plt.show()

# Correlation between season NRtg and L10 NRtg
r_season_l10, _ = stats.pearsonr(df.loc[common_idx, "delta_NRtg"].fillna(0),
                                   df.loc[common_idx, "delta_L10_NRtg"].fillna(0))
print(f"Correlation between delta_NRtg and delta_L10_NRtg: r={r_season_l10:.3f}")
print("(Low correlation confirms L10 adds independent signal)")
